In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
files1 = glob.glob('../../data/regression_outputs_new/Sampling/*/*/Fuse/Multi_Concat_pcahierachy/results.csv')
files2 = glob.glob('../../data/regression_outputs_new/Sampling/*/*/Fuse/Multi_Concat_pcahierachy_top1/results.csv')
files = files1 + files2
cities = []
for file in files:
    city = file.split('/')[-4]
    country = file.split('/')[-5]
    cities.append((city, country))
cities = list(set(cities))
len(cities)

In [ ]:
def load_data(city, country):
    file = f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy/results.csv'
    if not os.path.exists(file):
        file = f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy_top1/results.csv'
    df = pd.read_csv(file)
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    # if country == 'China':
    df = pd.merge(df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    # else:
    #     df = pd.merge(df, labels_sdg[['Code', 'SDG']], left_on='target', right_on='Code')
    return df

In [ ]:
data_all = {}
for city, country in cities:
    if city != 'All':
        data_all[city] = load_data(city, country)
    else:
        data_all[f'Selected cities in {country}'] = load_data(city, country)

In [ ]:
cities_df = pd.DataFrame(cities, columns=['city', 'country'])
cities_df.sort_values(['country', 'city'], inplace=True)
cities_df.reset_index(drop=True, inplace=True)
cities_df

In [ ]:
cities_df.loc[10, 'city'] = 'Selected cities in France'
cities_df.loc[12, 'city'] = 'Selected cities in Portugal'
cities_df

In [ ]:
n_rows = 5
n_cols = 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 20))

sdg_colors = {
    1: "#e5243b",
    3: "#4C9F38",
    4: "#C5192D",
    5: "#FF3A21",
    6: "#26BDE2",
    8: "#A21942",
    9: "#FD6925",
    10: "#DD1367",
    11: "#FD9D24",
    13: "#3F7E44",
    16: "#00689D"
}

for j, row in cities_df.iterrows():
    city = row['city']
    country = row['country']
    df = data_all[city]
    df['ratio_percent'] = df['ratio'] * 100
    
    ax = axes[j // n_cols, j % n_cols]

    # Build a palette covering the SDGs that appear in this city.
    custom_order = sorted(df['SDG'].unique())
    custom_palette = {sdg: sdg_colors.get(sdg, '#999999') for sdg in custom_order}

    # Scatter plot with custom colors.
    sns.scatterplot(data=df, x='ratio_percent', y='R2', hue='SDG', ax=ax,
                    alpha=0.6, palette=custom_palette, legend=False)

    for i, topic in enumerate(custom_order):
        data = df[df['SDG'] == topic]
        x = data['ratio'].values
        y_r2 = data['R2'].values    

        params_r2, covariance = curve_fit(log_fit, x, y_r2)
        perr_r2 = np.sqrt(np.diag(covariance))
        ci_upper = params_r2 + 1.96 * perr_r2
        ci_lower = params_r2 - 1.96 * perr_r2

        color = sdg_colors.get(topic, '#999999')

        x_fit = np.linspace(0, 1, 1000)
        y_fit_r2 = log_fit(x_fit, *params_r2)
        ax.plot(x_fit*100, y_fit_r2, label=f'SDG {topic}', color=color)

        y_fit_r2_upper = log_fit(x_fit, *ci_upper)
        y_fit_r2_lower = log_fit(x_fit, *ci_lower)
        ax.fill_between(x_fit*100, y_fit_r2_lower, y_fit_r2_upper, color=color, alpha=0.3)

        y_dot = 0.8
        x_080 = np.exp((0.8 - params_r2[0]) / params_r2[1])
        ax.scatter(x_080*100, 0.8, color=color, zorder=5)
        ax.axvline(x_080*100, color=color, linestyle='--', linewidth=1)
        ax.annotate(f'SDG {topic} = {x_080*100:.2f}%', 
                    xy=(x_080*100, log_fit(x_080, *params_r2)),
                    xytext=((x_080-0.05)*100, (0.05+i*0.1)),
                    fontsize=14,
                    color=color,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7)
                    )

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1)
    ax.legend().set_visible(False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)

    ax.set_title(f'{city}', fontsize=18)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    if j in [0, 4, 8, 12, 16]:
        ax.set_ylabel('$R^2$', fontsize=14)
    if j in [16, 17, 18, 19]:
        ax.set_xlabel('Sampling ratio (%)', fontsize=14)

# Drop empty subplot slots.
for j in range(n_rows * n_cols):
    if j >= len(cities_df):
        fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.savefig("../../data/figure_assets/S_strategic_sampling.pdf", dpi=300)
plt.savefig("../../data/figure_assets/S_strategic_sampling.png")
plt.show()